# Train on Colab

Runs `python -m fastmri_knee.train` on a Colab GPU and leaves everything on Drive:
`best.ckpt`, `last.ckpt`, `run_summary.json` and an MLflow file store. Register the result locally with
`make register CKPT=... SUMMARY=...` (K3). Steps and troubleshooting: `docs/training-colab.md`.

The kernel is remote. It cannot see local files or uncommitted changes, so it installs the package from a **pushed commit**.
Re-running the training cell after a disconnect resumes from `last.ckpt` in the same checkpoint dir.

In [ ]:
# --- Parameters: edit this cell only ---
GIT_SHA = ""  # full 40-char sha of a PUSHED commit (git rev-parse HEAD)
REPO = "mzdwedar/fastmri"

DRIVE_ROOT = "/content/drive/MyDrive/fastmri"  # expects {train,val}/ with .h5 files
RUN_NAME = "smoke"  # outputs go to DRIVE_ROOT/runs/RUN_NAME

NUM_FILES = 2  # .h5 files copied per split to local disk; 0 = all
NUM_SAMPLES = 64  # slices used per split; -1 = all (smoke run: small)
EPOCHS = 1
BATCH_SIZE = 8
NUM_WORKERS = 2

In [ ]:
# GPU check
import subprocess

print(
    subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
    or "NO GPU: Runtime > Change runtime type"
)

In [ ]:
# The commit must be pushed, otherwise pip cannot fetch it
import re
import urllib.error
import urllib.request

assert re.fullmatch(r"[0-9a-f]{40}", GIT_SHA), "set GIT_SHA to a full 40-char sha"
try:
    urllib.request.urlopen(f"https://api.github.com/repos/{REPO}/commits/{GIT_SHA}")
except urllib.error.HTTPError as exc:
    raise SystemExit(f"{GIT_SHA[:10]} not found on GitHub ({exc.code}): git push it first") from exc
print("commit is on GitHub")

In [ ]:
# Mount Drive. If this fails under the VS Code kernel, see "Drive fallback" in docs/training-colab.md
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

assert (Path(DRIVE_ROOT) / "train").is_dir() and (Path(DRIVE_ROOT) / "val").is_dir(), (
    f"no train/ and val/ in {DRIVE_ROOT}"
)

In [ ]:
# Colab ships Python 3.13; the package needs <3.13, so install into a uv-managed 3.12 venv
%pip install -q uv
!uv venv --python 3.12 /content/venv
!uv pip install -q --python /content/venv/bin/python "fastmri-knee[train] @ git+https://github.com/$REPO@$GIT_SHA"

In [ ]:
# Copy data to local disk: Drive's FUSE mount is slow for random h5 reads
import shutil

LOCAL = Path("/content/data")
for split in ("train", "val"):
    dst = LOCAL / split
    dst.mkdir(parents=True, exist_ok=True)
    files = sorted((Path(DRIVE_ROOT) / split).glob("*.h5"))
    for f in files[:NUM_FILES] if NUM_FILES else files:
        if not (dst / f.name).exists():
            shutil.copy(f, dst / f.name)
    print(split, len(list(dst.glob("*.h5"))), "files")

In [ ]:
# Train. Same RUN_NAME again = resume from last.ckpt (e.g. after a disconnect)
import os

out = Path(DRIVE_ROOT) / "runs" / RUN_NAME
env = {
    **os.environ,
    "GIT_SHA": GIT_SHA,  # run_summary.json provenance: there is no .git on the runtime
    "MPLBACKEND": "agg",  # Colab exports the inline backend, which the venv lacks
    "run_source": "colab",
    "train_root": str(LOCAL / "train"),
    "val_root": str(LOCAL / "val"),
    "data_root": str(LOCAL),
    "checkpoint_dir": str(out / "checkpoints"),
    "MLFLOW_TRACKING_URI": f"file:{out / 'mlruns'}",
    "TRAIN_RESULTS_FILE": str(out / "train_results.json"),
    "max_epochs": str(EPOCHS),
    "batch_size": str(BATCH_SIZE),
    "num_workers": str(NUM_WORKERS),
    "num_samples": str(NUM_SAMPLES),
}
out.mkdir(parents=True, exist_ok=True)
proc = subprocess.run(["/content/venv/bin/python", "-m", "fastmri_knee.train"], env=env, capture_output=True, text=True)
print((proc.stdout + proc.stderr)[-4000:])  # a bare subprocess hides its output from the cell
proc.check_returncode()

In [ ]:
# Result
import json

summary = json.loads((out / "checkpoints" / "run_summary.json").read_text())
print(json.dumps(summary, indent=2))